# V0.3 Durable Side Effect Lab

问题：文件已经修改但程序突然崩了，为什么不会重复修改？

这个实验展示：PREPARE -> DISPATCH -> filesystem effect -> crash before COMMIT -> restart -> hash inspection -> reconcile。

In [ ]:
MODE = "deterministic"

from pathlib import Path
import sys

def find_agentkernel_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "agentkernel").is_dir() and (path / "labs").is_dir():
            return path
    raise RuntimeError("Run this notebook from the AgentKernel repo root or the labs directory.")

REPO_ROOT = find_agentkernel_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from labs import create_lab

lab = create_lab("v03", mode=MODE)


## Step 1: Setup workspace and Session

In [ ]:
_ = lab.setup()

## Step 2: Inspect original file

In [ ]:
_ = lab.show_initial_state()

## Step 3: Model proposes apply_patch

这里的 deterministic path 是 scripted model decision；Kernel 执行仍然是真的。

In [ ]:
_ = lab.model_step()

## Step 4: PREPARE

Kernel 在真正修改文件前，先记录准备执行什么。

In [ ]:
_ = lab.prepare()

## Step 5: DISPATCH

Kernel 记录操作即将越过副作用边界。

In [ ]:
_ = lab.dispatch()

## Step 6: Filesystem effect

下一格会真正修改 `calculator.py`。

In [ ]:
_ = lab.apply_effect()

## Crash boundary

**现在文件已经修改，但 COMMIT 尚未发生。**

这正是普通 retry 最容易重复副作用的位置。下一格模拟进程崩溃。

In [ ]:
_ = lab.crash()

## Step 7: Restart fresh runtime

In [ ]:
_ = lab.restart()

## Step 8: Recovery analysis

Kernel 会看到：PREPARE = yes，DISPATCH = yes，COMMIT = no。然后检查真实文件 hash。

In [ ]:
_ = lab.analyze()

## Step 9: Reconcile, do not retry

如果 hash 匹配，说明副作用已经发生。Kernel 补齐 COMMIT，而不是再次 apply_patch。

In [ ]:
_ = lab.reconcile()

## Summary

In [ ]:
_ = lab.summary()